# 03 — Response body (RSC wire format)

Load `response_body` from `mitm_http_captures` and explore the stream format.

Same component path as notebooks 01–02. Run from repo root.

In [1]:
from collections import Counter
from urllib.parse import parse_qs, urlparse

from pprint import pprint
from sqlalchemy import text

from core.db import SessionLocal

component_path = "/flagship-web/rsc-action/actions/component"
prefix = "com.linkedin.sdui.generated.jobseeker.dsl.impl."


def component_suffix(url: str) -> str:
    component_id = parse_qs(urlparse(url).query)["componentId"][0]
    return component_id[len(prefix) :] if component_id.startswith(prefix) else component_id


query = text("""
    SELECT request_url, response_body
    FROM mitm_http_captures
    WHERE response_body IS NOT NULL
    ORDER BY captured_at_ms DESC
""")

with SessionLocal() as session:
    rows = session.execute(query).fetchall()

component_rows = [
    (url, body)
    for url, body in rows
    if urlparse(url).path == component_path
]

print(len(rows), "rows with a response body")
print(len(component_rows), "rows on component path")
print(component_rows[0][1][:400])

2670 rows with a response body
326 rows on component path
1:I["030d6035cb3a997efb1cff7a008d2f89",[],"default"]
3:I["e9e5744c902fddb98f0eb62aee5d400a",[],"TracedComponent"]
4:I["f54a4d9f94904eb227a6c1307124edd6",[],"ClientComponent"]
5:I["b55b61101826bcdf8a734370f7480e4e",[],"VisibleItemsProvider"]
6:I["d1ec7326e0cbf63aa25ba5dfdf299a74",[],"TrackingScopeProvider"]
7:I["e6bc3be678bd0f74780350224832cc6c",[],"TriggerButton"]
9:I["85b20fca39223dffe536dd03122e


## not plain JSON

In [2]:
import json

sample = component_rows[0][1]

try:
    json.loads(sample)
    print("whole body parses as JSON")
except json.JSONDecodeError as e:
    print("whole body is not JSON:", e)

print(len(sample.splitlines()), "lines")

whole body is not JSON: Extra data: line 1 column 2 (char 1)
125 lines


## line format

Each line looks like `<chunk_id>:<data>`

In [3]:
for line in component_rows[0][1].splitlines()[:5]:
    chunk_id, data = line.split(":", 1)
    print("chunk_id:", repr(chunk_id), "data starts with:", data[:80])
    print("-" * 40)

chunk_id: '1' data starts with: I["030d6035cb3a997efb1cff7a008d2f89",[],"default"]
----------------------------------------
chunk_id: '3' data starts with: I["e9e5744c902fddb98f0eb62aee5d400a",[],"TracedComponent"]
----------------------------------------
chunk_id: '4' data starts with: I["f54a4d9f94904eb227a6c1307124edd6",[],"ClientComponent"]
----------------------------------------
chunk_id: '5' data starts with: I["b55b61101826bcdf8a734370f7480e4e",[],"VisibleItemsProvider"]
----------------------------------------
chunk_id: '6' data starts with: I["d1ec7326e0cbf63aa25ba5dfdf299a74",[],"TrackingScopeProvider"]
----------------------------------------


## chunk ids

In [5]:
line_counts = Counter()
all_chunk_ids = Counter()

for url, body in component_rows:
    lines = body.splitlines()
    line_counts[len(lines)] += 1
    for line in lines:
        chunk_id, _ = line.split(":", 1)
        all_chunk_ids[chunk_id] += 1

print("lines per response:")
pprint(line_counts.most_common(10))
print()
print("chunk ids across all responses:")
pprint(all_chunk_ids.most_common(20))

lines per response:
[(1, 104),
 (13, 66),
 (9, 33),
 (14, 30),
 (10, 12),
 (31, 11),
 (98, 8),
 (16, 7),
 (101, 6),
 (12, 6)]

chunk ids across all responses:
[('0', 326),
 ('1', 222),
 ('3', 222),
 ('4', 222),
 ('5', 222),
 ('6', 222),
 ('7', 222),
 ('2', 222),
 ('8', 222),
 ('9', 189),
 ('a', 177),
 ('b', 175),
 ('c', 169),
 ('d', 103),
 ('e', 73),
 ('f', 70),
 ('10', 63),
 ('12', 60),
 ('13', 60),
 ('17', 60)]


## chunk ids by componentId suffix

In [6]:
by_suffix = {}

for url, body in component_rows:
    suffix = component_suffix(url)
    chunk_ids = {line.split(":", 1)[0] for line in body.splitlines()}
    by_suffix.setdefault(suffix, Counter()).update(chunk_ids)

for suffix in sorted(by_suffix):
    print(suffix, sorted(by_suffix[suffix].keys()))
    print("-" * 40)

aboutTheCompanyForJobDetails ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '10a', '10b', '10c', '10d', '10e', '10f', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '11a', '11b', '11c', '11d', '11e', '11f', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '12a', '12b', '12c', '12d', '12e', '12f', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '13a', '13b', '13c', '13d', '13e', '13f', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '14a', '14b', '14c', '14d', '14e', '14f', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '15a', '15b', '15c', '15d', '15e', '15f', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '16a', '16b', '16c', '16d', '16e', '16f', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '17a', '17b', '17c', '17d', '17e', '17f', '18', '180', '18

## parse line data as JSON

In [7]:
parse_ok = 0
parse_fail = 0

for url, body in component_rows:
    for line in body.splitlines():
        _, data = line.split(":", 1)
        try:
            json.loads(data)
            parse_ok += 1
        except json.JSONDecodeError:
            parse_fail += 1

print(parse_ok, "lines parsed as JSON")
print(parse_fail, "lines failed JSON parse")

9503 lines parsed as JSON
2168 lines failed JSON parse


## RSC element shape

Lists that look like `['$', component_type, key, props_dict]`

In [ ]:
def is_rsc_node(value):
    return isinstance(value, list) and len(value) == 4 and value[0] == "$"


def find_rsc_nodes(obj, found=None):
    if found is None:
        found = []
    if is_rsc_node(obj):
        found.append(obj)
    elif isinstance(obj, dict):
        for v in obj.values():
            find_rsc_nodes(v, found)
    elif isinstance(obj, list):
        for item in obj:
            find_rsc_nodes(item, found)
    return found


# pick one line from the first response and inspect
line = component_rows[0][1].splitlines()[0]
chunk_id, data = line.split(":", 1)
try:
    parsed = json.loads(data)
except:
    parsed = data
    print("failed to parse", data)
print("chunk_id:", chunk_id)
print("top-level type:", type(parsed))
print(parsed if not is_rsc_node(parsed) else parsed[:3], "...")
print()

nodes = find_rsc_nodes(parsed)
print(len(nodes), "RSC nodes in this chunk")
if nodes:
    print("sample node component types:", Counter(n[1] for n in nodes).most_common(10))
    print("sample node:")
    pprint(nodes[0])

failed to parse I["030d6035cb3a997efb1cff7a008d2f89",[],"default"]
chunk_id: 1


NameError: name 'parsed' is not defined

## componentId suffixes with responses

In [9]:
pprint(Counter(component_suffix(url) for url, _ in component_rows).most_common())

[('aboutTheCompanyForJobDetails', 33),
 ('jobAlertToggle', 33),
 ('peopleWhoCanHelp', 33),
 ('premiumCompanyInsightsForJobDetails', 33),
 ('aboutTheJob', 33),
 ('resumeReview', 33),
 ('premiumApplicantInsightsForJobDetails', 33),
 ('similarJobs', 31),
 ('manageJobBanner', 31),
 ('howYouFitGuide', 30),
 ('jobMatch', 3)]
